# 🚀 Trip Classification – Competition Notebook (Macro F1 Oriented)

Notebook ini dikembangkan untuk **kompetisi klasifikasi perjalanan** dengan fokus utama pada **maksimalisasi Macro F1 Score**, khususnya pada kondisi **class imbalance** dan **pola non-linear kompleks** pada data sensor, trafik, dan ekonomi perjalanan.

---

## 🎯 Objective

Tujuan utama eksperimen ini adalah:
- Mengoptimalkan **Macro F1 Score** sebagai metrik evaluasi utama.
- Meningkatkan performa kelas minoritas tanpa mengorbankan stabilitas kelas mayoritas.
- Membangun pipeline yang **robust, reproducible, dan competition-ready**.

## 🧹 Data Cleaning & Memory Optimization

- Konversi tipe data numerik ke format paling efisien (`int8`, `int16`, `float32`) untuk:
  - Mengurangi penggunaan memori
  - Mempercepat proses training
- Pembersihan kolom `Battery_Level` dari karakter non-numerik.
- Nilai hilang **tidak dihapus**, tetapi dimodelkan sebagai **informative missingness** melalui flag biner.

> Prinsip: *Missingness is signal, not noise.*

In [1]:
import pandas as pd
import numpy as np
import gc, os, joblib, random
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, classification_report
from scipy.optimize import minimize

def set_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    print(f"-> Seed locked at: {seed}")

def reduce_mem_usage(df):
    for col in df.columns:
        col_type = df[col].dtype
        if col_type != object:
            c_min, c_max = df[col].min(), df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                else: df[col] = df[col].astype(np.int32)
            else: df[col] = df[col].astype(np.float32)
    return df

set_seed(42)
BASE_PATH = r"C:\Users\ADVAN\OneDrive - Universitas Diponegoro\Documents\Berkas\Lomba\DataVers 2026\Dataset"

-> Seed locked at: 42


## ⏱️ Temporal Features

- Konversi `Timestamp` ke format datetime.
- Ekstraksi fitur `hour` untuk menangkap:
  - Pola jam sibuk
  - Perjalanan malam
  - Perilaku operasional kendaraan

---

## 🌍 Geospatial Features

- Perhitungan jarak garis lurus menggunakan **Haversine Formula**.
- Pembentukan fitur:

In [2]:
def haversine_np(lat1, lon1, lat2, lon2):
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    d = np.sin((lat2-lat1)/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin((lon2-lon1)/2)**2
    return 6371 * 2 * np.arcsin(np.sqrt(d))

def apply_feature_interaction(df):
    # 1. Rasio Ekonomi
    df['Price_per_KM'] = df['Est_Price_IDR'] / (df['Distance_KM'] + 0.01)
    
    # 2. Operasional Stress
    traffic_map = {'Lancar': 1, 'Sedang': 2, 'Padat': 3, 'Macet': 4}
    t_numeric = df['Traffic'].map(traffic_map).fillna(2)
    surge_numeric = pd.to_numeric(df['Surge_Multiplier'], errors='coerce').fillna(1)
    df['Traffic_Pressure_Score'] = t_numeric * surge_numeric
    
    # 3. Sensor Stability & Route
    df['Sensor_Stability_Index'] = df['Accel_Z'] * (1 - df['is_missing_Gyro_Z'])
    df['Route_ID'] = df['Pickup_Zone'].astype(str) + "_" + df['Dropoff_Zone'].astype(str)
    
    return df

def ultimate_pipeline(df):
    # (Logika Cleaning & Geospasial Dasar)
    if 'Battery_Level' in df.columns and df['Battery_Level'].dtype == 'object':
        df['Battery_Level'] = pd.to_numeric(df['Battery_Level'].str.replace('%', ''), errors='coerce')
    
    # Missing Flags
    for col in ['GPS_Accuracy_M', 'Accel_Z', 'Gyro_Z']:
        df[f'is_missing_{col}'] = df[col].isnull().astype('int8')

    # Time Features
    df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')
    df['hour'] = df['Timestamp'].dt.hour.fillna(-1).astype('int8')
    
    # Geospatial
    dist_hav = haversine_np(df['Pickup_Lat'], df['Pickup_Long'], df['Dropoff_Lat'], df['Dropoff_Long'])
    df['Route_Circuity'] = (df['Distance_KM'] / (dist_hav + 0.1)).astype('float32')
    
    # Apply Interactions
    df = apply_feature_interaction(df)
    
    drop_cols = ['Pickup_Lat', 'Pickup_Long', 'Dropoff_Lat', 'Dropoff_Long', 'Timestamp', 'Device_FP']
    df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)
    return reduce_mem_usage(df)

## 📥 Data Loading, Encoding, and Train–Validation Split

Pada tahap ini dilakukan pemuatan data, pemisahan fitur dan target, encoding variabel kategorikal, serta pembagian data latih dan validasi dengan rasio **90:10**.

---

### 1️⃣ Data Loading & Feature Processing

- Data **train** dan **test** dimuat dari file CSV.
- Keduanya diproses menggunakan fungsi `ultimate_pipeline` untuk menjamin:
  - Konsistensi preprocessing
  - Tidak adanya data leakage
  - Struktur fitur yang identik antara train dan test

---

### 2️⃣ Target Encoding

- Variabel target `Trip_Label` diencoding menggunakan **LabelEncoder**.
- Label kosong atau tidak terdefinisi diisi sebagai `"Unknown"` sebelum encoding.
- Pendekatan ini menghasilkan representasi numerik target yang kompatibel dengan LightGBM.

---

### 3️⃣ Feature–Target Separation

- Kolom identitas (`Trip_ID`) dan target (`Trip_Label`) dikeluarkan dari fitur input.
- `Trip_ID` pada data test disimpan terpisah untuk keperluan submission.

---

### 4️⃣ Categorical Encoding Strategy

- Fitur kategorikal yang digunakan:
  - `Pickup_Zone`, `Dropoff_Zone`
  - `Promo_Code`, `Car_Model`
  - `Payment_Method`, `Weather`
  - `Traffic`, `Signal_Strength`
  - `Route_ID`
- Setiap fitur kategorikal diencoding menggunakan **Label Encoding**.
- Encoder dilatih pada **gabungan data train dan test** untuk:
  - Mencegah unseen category saat inference
  - Menjaga konsistensi mapping kategori
- Seluruh hasil encoding dikonversi ke `int32` untuk efisiensi memori dan performa.

---

### 5️⃣ Train–Validation Split (90/10)

- Data dibagi dengan rasio **90% train** dan **10% validation**.
- Parameter `stratify=y` digunakan untuk:
  - Menjaga distribusi kelas tetap seimbang
  - Menghasilkan evaluasi Macro F1 yang stabil dan representatif
- `random_state=42` memastikan eksperimen bersifat reproducible.

---

Tahap ini menghasilkan dataset yang **siap digunakan untuk training model**, bebas dari data leakage, dan selaras dengan tujuan optimasi **Macro F1 Score**.


In [3]:
# --- Load & Process ---
train = ultimate_pipeline(pd.read_csv(os.path.join(BASE_PATH, 'train.csv')))
test = ultimate_pipeline(pd.read_csv(os.path.join(BASE_PATH, 'test.csv')))

# --- Encoding ---
le_target = LabelEncoder()
y = le_target.fit_transform(train['Trip_Label'].fillna('Unknown'))
X = train.drop(columns=['Trip_Label', 'Trip_ID'])
test_ids = test['Trip_ID']
X_test = test.drop(columns=['Trip_ID'])

cat_cols = ['Pickup_Zone', 'Dropoff_Zone', 'Promo_Code', 'Car_Model', 'Payment_Method', 'Weather', 'Traffic', 'Signal_Strength', 'Route_ID']
for col in cat_cols:
    le = LabelEncoder()
    full_vals = pd.concat([X[col], X_test[col]])
    le.fit(full_vals)
    X[col] = le.transform(X[col]).astype('int32')
    X_test[col] = le.transform(X_test[col]).astype('int32')

# --- Split 90/10 ---
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.1, stratify=y, random_state=42)
gc.collect()

46

## 🌲 Model Training & Macro F1 Optimization

Model utama dilatih menggunakan **LightGBM Classifier** dengan parameter yang disesuaikan untuk menangani **class imbalance** dan pola non-linear kompleks. Training dilakukan pada 90% data dengan **early stopping** berbasis validation set untuk mencegah overfitting.

Setelah training, dihasilkan **probabilitas prediksi per kelas** pada validation set. Probabilitas ini kemudian dioptimasi menggunakan pendekatan **post-hoc probability re-weighting** dengan algoritma **Nelder–Mead**, di mana bobot kelas diatur untuk memaksimalkan **Macro F1 Score**.

Pendekatan ini memungkinkan peningkatan performa Macro F1 **tanpa retraining model**, serta membantu memperbaiki recall kelas minoritas secara efektif.

In [4]:
# 1. Training Base Model
clf_base = lgb.LGBMClassifier(
    n_estimators=3000, 
    learning_rate=0.02, 
    num_leaves=255, 
    class_weight='balanced', 
    random_state=42, n_jobs=-1
)

clf_base.fit(
    X_train, y_train, 
    eval_set=[(X_val, y_val)], 
    callbacks=[lgb.early_stopping(100), lgb.log_evaluation(100)]
)

# 2. Nelder-Mead Optimization
val_probs = clf_base.predict_proba(X_val)

def macro_f1_objective(w, y_true, probs):
    preds = np.argmax(probs * w, axis=1)
    return -f1_score(y_true, preds, average='macro')

res = minimize(macro_f1_objective, x0=[1.0]*len(le_target.classes_), args=(y_val, val_probs), method='Nelder-Mead')
best_weights = res.x

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.451555 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4232
[LightGBM] [Info] Number of data points in the train set: 7200000, number of used features: 26
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Info] Start training from score -1.609438
Training until validation scores don't improve for 100 rounds
[100]	valid_0's multi_logloss: 0.921035
[200]	valid_0's multi_logloss: 0.871599
[300]	valid_0's multi_logloss: 0.863601
[400]	valid_0's multi_logloss: 0.861551
[500]	valid_0's multi_logloss: 0.860595
[600]	valid_0's multi_logloss: 0.859861
[700]	valid_0's multi_logloss: 0.859174
[800]	valid_0's multi_logloss: 0.858496
[900]	valid_0's multi_logloss: 0.857856
[1000]	valid_0's 

## 📊 Model Evaluation

Evaluasi dilakukan pada validation set (10%) dengan membandingkan performa:

- **Macro F1 (Raw)**: prediksi langsung dari model tanpa optimasi  
- **Macro F1 (Optimized)**: prediksi setelah penerapan bobot kelas hasil Nelder–Mead  

Selain nilai Macro F1 dan *net gain*, ditampilkan **classification report** untuk menganalisis performa tiap kelas secara detail, dengan fokus pada peningkatan recall dan keseimbangan antar kelas.


In [5]:
print("\n=== HASIL EVALUASI MODEL 90% ===")
val_preds_raw = np.argmax(val_probs, axis=1)
val_preds_opt = np.argmax(val_probs * best_weights, axis=1)

f1_raw = f1_score(y_val, val_preds_raw, average='macro')
f1_opt = f1_score(y_val, val_preds_opt, average='macro')

print(f"-> Macro F1 (Raw)      : {f1_raw:.5f}")
print(f"-> Macro F1 (Optimized): {f1_opt:.5f}")
print(f"-> Net Gain            : {f1_opt - f1_raw:.5f}")

print("\n[Classification Report]:")
print(classification_report(y_val, val_preds_opt, target_names=le_target.classes_))


=== HASIL EVALUASI MODEL 90% ===
-> Macro F1 (Raw)      : 0.57780
-> Macro F1 (Optimized): 0.59044
-> Net Gain            : 0.01263

[Classification Report]:
                   precision    recall  f1-score   support

 Fraud_Indication       0.99      0.98      0.99     40031
 Navigation_Issue       0.13      0.14      0.14     80179
     Perfect_Trip       0.72      0.71      0.72    439761
 Safety_Violation       0.99      0.95      0.97    160160
Service_Complaint       0.13      0.15      0.14     79869

         accuracy                           0.66    800000
        macro avg       0.59      0.59      0.59    800000
     weighted avg       0.67      0.66      0.67    800000



## 📦 Model Saving & Submission Export

Model akhir disimpan sebagai satu paket yang mencakup:
- Trained LightGBM model
- Bobot kelas hasil optimasi Nelder–Mead
- Daftar fitur
- Label encoder target

Untuk submission, prediksi pada data test dihasilkan menggunakan **probabilitas yang telah dioptimasi**, kemudian diekspor ke file CSV sesuai format lomba.

In [6]:
# 1. Simpan Model Pack
model_pack = {
    'model': clf_base,
    'best_weights': best_weights,
    'features': X_train.columns.tolist(),
    'le_target': le_target
}
joblib.dump(model_pack, 'lgb_model_90_optimized.pkl')

# 2. Export Submission (Versi 90%)
test_probs_90 = clf_base.predict_proba(X_test)
final_preds_90 = np.argmax(test_probs_90 * best_weights, axis=1)

submission = pd.DataFrame({
    'Trip_ID': test_ids, 
    'Trip_Label': le_target.inverse_transform(final_preds_90)
})
submission.to_csv('submission_model_90_only.csv', index=False)
print("\n-> File 'submission_model_90_only.csv' telah siap.")


-> File 'submission_model_90_only.csv' telah siap.
